In [4]:
import pandas as pd
import os
import csv

# --- Configuration ---
# --- Pandas Display Options (Add these lines) ---
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)      # Set a wider display width to prevent truncation
pd.set_option('display.max_rows', None)   # Display all rows (if matching_matches has many rows)
# pd.set_option('display.colheader_justify', 'left') # Optional: Adjust column header alignment
# Set the directory for your main Premier League data
DATA_DIR = './LiveSum_++/english-premier-league/' 
print(f"Using main data directory: {DATA_DIR}")

# Set the directory for your individual match stat files (e.g., sample_1_table.csv)
SAMPLE_DATA_DIR = './LiveSum_++/training_data/'
print(f"Using sample data directory: {SAMPLE_DATA_DIR}")

# List of CSV files to process for the main historical data
CSV_FILES = [f'{year}-{str(int(year) + 1)[-2:]}.csv' 
             for year in range(2013, 2022)] # Generates '2013-14.csv', '2014-15.csv', etc. up to '2021-22.csv'
print(f"CSV files to process (main data): {CSV_FILES}")

# --- Dynamic TARGET_STATS Population Function ---
def populate_target_stats_from_csv(file_number, data_directory):
    """
    Populates the TARGET_STATS dictionary dynamically from a single CSV file
    with 'Away Team' and 'Home Team' data on separate rows.

    Args:
        file_number (int): The number of the CSV file to read (e.g., 1 for sample_1_table.csv).
        data_directory (str): The directory where the CSV files are located (e.g., SAMPLE_DATA_DIR).

    Returns:
        tuple: A tuple containing the dynamically populated TARGET_STATS dictionary and 
               a list of column names extracted from the sample CSV's header (excluding 'Team').
               Returns (None, None) if the file/data is not found or an error occurs.
    """
    file_name = f"sample_{file_number}_table.csv"
    file_path = os.path.join(data_directory, file_name)
    
    dynamic_target_stats = {}
    sample_columns = []

    try:
        with open(file_path, mode='r') as csvfile:
            reader = csv.reader(csvfile)
            
            # Read the header row and strip whitespace
            header = [h.strip() for h in next(reader)] 
            
            # Find the indices of the columns we need
            column_indices = {}
            for i, col_name in enumerate(header):
                column_indices[col_name] = i
                if col_name != 'Team': # Collect column names from sample, excluding 'Team'
                    sample_columns.append(col_name)

            if 'Team' not in column_indices:
                print(f"Error: 'Team' column not found in {file_name}. Skipping dynamic stats population.")
                return None, None

            away_data = None
            home_data = None

            # Read the next two rows (Away Team and Home Team)
            try:
                row1 = [d.strip() for d in next(reader)]
                row2 = [d.strip() for d in next(reader)]
            except StopIteration:
                print(f"Error: Not enough data rows (expected 2) in {file_name} after header. Skipping dynamic stats population.")
                return None, None

            # Determine which row is Away and which is Home
            team_col_idx = column_indices['Team']
            if row1[team_col_idx].lower() == 'away team' and row2[team_col_idx].lower() == 'home team':
                away_data = row1
                home_data = row2
            elif row1[team_col_idx].lower() == 'home team' and row2[team_col_idx].lower() == 'away team':
                away_data = row2
                home_data = row1
            else:
                print(f"Error: Unexpected team names in rows after header in {file_name}. Expected 'Away Team' and 'Home Team'. Skipping dynamic stats population.")
                return None, None
            
            # Helper to safely get integer value
            def get_stat_value(data_row, stat_name_in_csv):
                if stat_name_in_csv in column_indices:
                    try:
                        return int(data_row[column_indices[stat_name_in_csv]])
                    except ValueError:
                        print(f"Warning: Could not convert '{data_row[column_indices[stat_name_in_csv]]}' for {stat_name_in_csv} in '{data_row[team_col_idx]}' to integer. Skipping this stat.")
                        return None
                return None

            # Populate dynamic_target_stats for Away Team
            goals_away = get_stat_value(away_data, 'Goals')
            if goals_away is not None: dynamic_target_stats['FTAG'] = {'operator': '==', 'value': goals_away}
            shots_away = get_stat_value(away_data, 'Shots')
            if shots_away is not None: dynamic_target_stats['AS'] = {'operator': '==', 'value': shots_away}
            fouls_away = get_stat_value(away_data, 'Fouls')
            if fouls_away is not None: dynamic_target_stats['AF'] = {'operator': '==', 'value': fouls_away}
            yellow_away = get_stat_value(away_data, 'Yellow Cards')
            if yellow_away is not None: dynamic_target_stats['AY'] = {'operator': '==', 'value': yellow_away}
            red_away = get_stat_value(away_data, 'Red Cards')
            if red_away is not None: dynamic_target_stats['AR'] = {'operator': '==', 'value': red_away}
            corners_away = get_stat_value(away_data, 'Corner Kicks')
            if corners_away is not None: dynamic_target_stats['AC'] = {'operator': '==', 'value': corners_away}
            # Note: Add 'Free Kicks' and 'Offsides' if you want them from the sample file and they exist in the main data
            # free_kicks_away = get_stat_value(away_data, 'Free Kicks')
            # if free_kicks_away is not None: dynamic_target_stats['AFK'] = {'operator': '==', 'value': free_kicks_away}
            # offsides_away = get_stat_value(away_data, 'Offsides')
            # if offsides_away is not None: dynamic_target_stats['AO'] = {'operator': '==', 'value': offsides_away}


            # Populate dynamic_target_stats for Home Team
            goals_home = get_stat_value(home_data, 'Goals')
            if goals_home is not None: dynamic_target_stats['FTHG'] = {'operator': '==', 'value': goals_home}
            shots_home = get_stat_value(home_data, 'Shots')
            if shots_home is not None: dynamic_target_stats['HS'] = {'operator': '==', 'value': shots_home}
            fouls_home = get_stat_value(home_data, 'Fouls')
            if fouls_home is not None: dynamic_target_stats['HF'] = {'operator': '==', 'value': fouls_home}
            yellow_home = get_stat_value(home_data, 'Yellow Cards')
            if yellow_home is not None: dynamic_target_stats['HY'] = {'operator': '==', 'value': yellow_home}
            red_home = get_stat_value(home_data, 'Red Cards')
            if red_home is not None: dynamic_target_stats['HR'] = {'operator': '==', 'value': red_home}
            corners_home = get_stat_value(home_data, 'Corner Kicks')
            if corners_home is not None: dynamic_target_stats['HC'] = {'operator': '==', 'value': corners_home}
            # Note: Add 'Free Kicks' and 'Offsides' if you want them from the sample file and they exist in the main data
            # free_kicks_home = get_stat_value(home_data, 'Free Kicks')
            # if free_kicks_home is not None: dynamic_target_stats['HFK'] = {'operator': '==', 'value': free_kicks_home}
            # offsides_home = get_stat_value(home_data, 'Offsides')
            # if offsides_home is not None: dynamic_target_stats['HO'] = {'operator': '==', 'value': offsides_home}

            return dynamic_target_stats, sample_columns

    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found. Skipping dynamic stats population.")
        return None, None
    except Exception as e:
        print(f"An unexpected error occurred while reading {file_path}: {e}. Skipping dynamic stats population.")
        return None, None
## Core Data Loading and Filtering Functions
def load_and_combine_data(data_directory, csv_files, columns_to_keep=None):
    """
    Loads multiple CSV files from a directory into a single Pandas DataFrame.
    Assumes CSVs contain Premier League match data.
    Optionally drops columns not in `columns_to_keep`.
    """
    all_data = []
    for filename in csv_files:
        filepath = os.path.join(data_directory, filename)
        if os.path.exists(filepath):
            try:
                df = pd.read_csv(filepath, encoding='latin1') 
                df['Season'] = filename.split('.')[0] # Add a 'Season' column
                
                # --- New Logic: Drop columns not in sample_columns ---
                if columns_to_keep:
                    # Identify columns in the current DataFrame that are NOT in columns_to_keep
                    cols_to_drop = [col for col in df.columns if col not in columns_to_keep and col != 'Season']
                    if cols_to_drop:
                        df = df.drop(columns=cols_to_drop)
                        print(f"Dropped columns from {filename}: {cols_to_drop}")
                # --- End New Logic ---

                all_data.append(df)
                print(f"Successfully loaded: {filename}")
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        else:
            print(f"File not found: {filename}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        print("No data loaded. Please check your DATA_DIR and CSV_FILES list.")
        return pd.DataFrame()

def find_matches_by_stats(dataframe, target_stats):
    """
    Filters a DataFrame to find matches that meet the specified statistical criteria.
    """
    if dataframe.empty:
        print("Input DataFrame is empty. Cannot filter.")
        return pd.DataFrame()

    # Start with a condition where all rows are True
    condition = pd.Series(True, index=dataframe.index)

    for col, criteria in target_stats.items():
        if col in dataframe.columns:
            operator = criteria['operator']
            value = criteria['value']
            
            if operator == '>':
                condition &= (dataframe[col] > value)
            elif operator == '<':
                condition &= (dataframe[col] < value)
            elif operator == '==':
                condition &= (dataframe[col] == value)
            elif operator == '>=':
                condition &= (dataframe[col] >= value)
            elif operator == '<=':
                condition &= (dataframe[col] <= value)
            elif operator == '!=':
                condition &= (dataframe[col] != value)
            else:
                print(f"Unsupported operator '{operator}' for column '{col}'. Skipping this condition.")
        else:
            print(f"Column '{col}' not found in DataFrame. Skipping this condition.")
            
    return dataframe[condition]

# --- End of Configuration ---
## Execution Logic
if __name__ == "__main__":
    # --- Step 1: Get user input for the sample file number ---
    input_file_number = None
    while input_file_number is None:
        try:
            input_file_number = int(input("Enter the sample file number to get target stats from (e.g., 1 for sample_1_table.csv): "))
        except ValueError:
            print("Invalid input. Please enter an integer.")

    # --- Step 2: Dynamically populate TARGET_STATS and get sample columns ---
    print(f"\nAttempting to populate TARGET_STATS from sample_{input_file_number}_table.csv...")
    dynamic_target_stats, sample_cols_from_file = populate_target_stats_from_csv(input_file_number, SAMPLE_DATA_DIR)

    if dynamic_target_stats:
        print("\nDynamically Populated TARGET_STATS:")
        for stat, conditions in dynamic_target_stats.items():
            print(f"'{stat}': {{'operator': '{conditions['operator']}', 'value': {conditions['value']}}}")
        
        # Now use the dynamically populated stats for filtering the main data
        FINAL_TARGET_STATS = dynamic_target_stats
    else:
        print("\nCould not populate TARGET_STATS dynamically. Falling back to default TARGET_STATS if defined, or exiting.")
        FINAL_TARGET_STATS = {} # Or your initial hardcoded TARGET_STATS if you want a fallback

    # --- Step 3: Load and combine the main Premier League data, dropping irrelevant columns ---
    print("\nLoading and combining main Premier League data, dropping columns not in sample...")
    
    # We need to map the generic sample column names to the specific column names used in the main dataframes.
    # Based on your initial comments:
    # 'Goals' (from sample) -> 'FTHG'/'FTAG' (in main, handled by TARGET_STATS keys)
    # 'Shots' (from sample) -> 'HS'/'AS' (in main)
    # 'Fouls' (from sample) -> 'HF'/'AF' (in main)
    # 'Yellow Cards' (from sample) -> 'HY'/'AY' (in main)
    # 'Red Cards' (from sample) -> 'HR'/'AR' (in main)
    # 'Corner Kicks' (from sample) -> 'HC'/'AC' (in main)
    # 'Free Kicks' (from sample) -> typically not in main data as 'FK'
    # 'Offsides' (from sample) -> 'HO'/'AO' (in main)

    # Let's create a list of *expected* columns in the main data based on the sample.
    # This list will be used to tell load_and_combine_data what to keep.
    # It's important to include "HomeTeam", "AwayTeam", "FTR" (Full Time Result), "Div", "Date"
    # and any other non-stat columns you always want to retain.
    
    # Create a set of columns expected in the main dataframe, derived from the sample's relevant columns
    # and common match identifiers.
    main_df_relevant_columns = set([
        'HomeTeam', 'AwayTeam', 'FTR', 'Div', 'Date', 'Referee', # Common match identifiers
        'FTHG', 'FTAG', 'HS', 'AS', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'HC', 'AC', 'HTHG', 'HTAG', 'HTR'
    ])

    # Pass this set of relevant columns to the load function.
    combined_football_data = load_and_combine_data(DATA_DIR, CSV_FILES, columns_to_keep=main_df_relevant_columns)

    # --- Step 4: Find matches using the (dynamically) defined TARGET_STATS ---
    if not combined_football_data.empty and FINAL_TARGET_STATS:
        print(f"\nTotal rows loaded: {len(combined_football_data)}")
        print("\nFirst 5 rows of combined main data (after column dropping):")
        # Ensure display() is available if running outside Jupyter/Colab
        try:
            from IPython.display import display
        except ImportError:
            # If not in IPython, just use print
            def display(df):
                print(df.to_string()) # to_string() ensures full display in console
        
        display(combined_football_data.head()) 

        print("\nSearching for matches based on the following criteria:")
        for stat, conditions in FINAL_TARGET_STATS.items():
            print(f"  - {stat} {conditions['operator']} {conditions['value']}")

        matching_matches = find_matches_by_stats(combined_football_data, FINAL_TARGET_STATS)

        print(f"\nFound {len(matching_matches)} match(es) matching the specified stats across all seasons:")
        display(matching_matches) # This will now display the full rows/columns
    elif not FINAL_TARGET_STATS:
        print("No target statistics were defined. Cannot filter main data.")
    else:
        print("No main data to process. Please check the data loading step.")

Using main data directory: ./LiveSum_++/english-premier-league/
Using sample data directory: ./LiveSum_++/training_data/
CSV files to process (main data): ['2013-14.csv', '2014-15.csv', '2015-16.csv', '2016-17.csv', '2017-18.csv', '2018-19.csv', '2019-20.csv', '2020-21.csv', '2021-22.csv']

Attempting to populate TARGET_STATS from sample_49_table.csv...

Dynamically Populated TARGET_STATS:
'FTAG': {'operator': '==', 'value': 0}
'AS': {'operator': '==', 'value': 15}
'AF': {'operator': '==', 'value': 6}
'AY': {'operator': '==', 'value': 1}
'AR': {'operator': '==', 'value': 0}
'AC': {'operator': '==', 'value': 14}
'FTHG': {'operator': '==', 'value': 0}
'HS': {'operator': '==', 'value': 7}
'HF': {'operator': '==', 'value': 7}
'HY': {'operator': '==', 'value': 2}
'HR': {'operator': '==', 'value': 0}
'HC': {'operator': '==', 'value': 4}

Loading and combining main Premier League data, dropping columns not in sample...
Dropped columns from 2013-14.csv: ['HST', 'AST', 'B365H', 'B365D', 'B365A'

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HF,AF,HC,AC,HY,AY,HR,AR,Season
0,E0,17/08/13,Arsenal,Aston Villa,1,3,A,1,1,D,A Taylor,16,9,15,18,4,3,4,5,1,0,2013-14
1,E0,17/08/13,Liverpool,Stoke,1,0,H,1,0,H,M Atkinson,26,10,11,11,12,6,1,1,0,0,2013-14
2,E0,17/08/13,Norwich,Everton,2,2,D,0,0,D,M Oliver,8,19,13,10,6,8,2,0,0,0,2013-14
3,E0,17/08/13,Sunderland,Fulham,0,1,A,0,0,D,N Swarbrick,20,5,14,14,6,1,0,3,0,0,2013-14
4,E0,17/08/13,Swansea,Man United,1,4,A,0,2,A,P Dowd,17,15,13,10,7,4,1,3,0,0,2013-14



Searching for matches based on the following criteria:
  - FTAG == 0
  - AS == 15
  - AF == 6
  - AY == 1
  - AR == 0
  - AC == 14
  - FTHG == 0
  - HS == 7
  - HF == 7
  - HY == 2
  - HR == 0
  - HC == 4

Found 1 match(es) matching the specified stats across all seasons:


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HF,AF,HC,AC,HY,AY,HR,AR,Season
244,E0,08/02/14,Norwich,Man City,0,0,D,0,0,D,J Moss,7,15,7,6,4,14,2,1,0,0,2013-14
